<a href="https://colab.research.google.com/github/nikradaev2004-lgtm/Python_MIPT_10-2026/blob/main/Bash_Radaev_28_10_2026.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Bash. 28/X/2026

Создайте дерево work/ из лекции одной командой mkdir -p с фигурными скобками

In [ ]:
mkdir -p work/{genomes/species,reads/sample} && touch work/genomes/species/species.fasta work/reads/sample/{sample_1.fastq,sample_2.fastq}

Посчитайте последовательности в FASTA и риды в FASTQ


In [ ]:
# Подсчет последовательностей в fasta
grep -c '^>' example_fasta.fasta

# Подсчет ридов в fastq (строки / 4)
$(cat example_fastq.fastq | wc -l) / 4

Напишите count_reads.sh: для каждого *.fastq.gz в папке-аргументе — имя и число ридов; строгий режим, проверка аргумента

In [ ]:
function(){
    local folder="$1"
    for f in "$folder"/*.fastq.gz; do
        echo "$f"
        n=$(( $(zcat "$f" | wc -l) / 4 ))
        echo "$n"
    done
}

Попросите ИИ улучшить его: вывод в TSV и parallel. Прогоните ShellCheck и объясните каждую строку


In [ ]:
#!/bin/bash # стандартный заголовок в Bash

function count_reads() { # Создание функции count_reads
    local folder="$1" # создание локальной переменной - в нее сообщается папка
    local f # создание локальной переменной f. Она может использованиться для создания цикла и получения имени конеретного файла
    local n # создание локальной переменной n. В дальнейшем в нее будет сообщено количество ридов в файле после расчета

    if [[ -z "$folder" || ! -d "$folder" ]]; then # условие проверяет, чтобы указанная папка действительно существовала. Если она существует, код продолжает работать
        echo "Ошибка: укажите существующую папку" >&2 # если папки не существует, то выводится данный текст
        return 1
    fi

    for f in "$folder"/*.fastq.gz; do # для всех файлов, которые оканчиваются на .fastq.gz
        [[ -f "$f" ]] || continue # продолжить выполение цикла

        n=$(gzip -cd "$f" | awk 'END { print int(NR / 4) }') # рассчитать количество ридов в каждом файле. Присвоить значение переменной n

        echo "$f" # вывести переменную f, то есть название файла
        echo "$n" # вывести количество ридов в каждом файле (переменную n)
    done # окончить цикл
}

Перепишите его на Python и сравните: длина, скорость, читаемость


In [ ]:
import gzip # необходимо импортировать пакет gzip (в предыдущем скрипте на Bash импорт не нужен)
from pathlib import Path # необходимо испортировать модуль Path (в предыдущем скрипте на Bash импорт не нужен)

def count_reads(folder):
    folder = Path(folder)

    if not folder.is_dir():
        print("Ошибка: укажите существующую папку")
        return 1

    for f in folder.glob("*.fastq.gz"):
        with gzip.open(f, "rt") as file:
            n = sum(1 for _ in file) // 4

        print(f)
        print(n)

# Длина (количество строк): Bash: 16; Python: 12
# Читаемость: одинаковая
# Скорость: без существенных отличий